In [163]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [164]:
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "cardiovascular" / "cardio_data_processed.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DATA_PATH)
print("Processed data:", PROCESSED_DIR)

Raw data: ../data/raw/cardiovascular/cardio_data_processed.csv
Processed data: ../data/processed


In [165]:
df = pd.read_csv(RAW_DATA_PATH)

print("Shape:", df.shape)

Shape: (68205, 17)


In [166]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,bmi,bp_category,bp_category_encoded
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50,21.967120,Hypertension Stage 1,Hypertension Stage 1
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55,34.927679,Hypertension Stage 2,Hypertension Stage 2
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1,51,23.507805,Hypertension Stage 1,Hypertension Stage 1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48,28.710479,Hypertension Stage 2,Hypertension Stage 2
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0,47,23.011177,Normal,Normal


In [167]:
data = df.copy()

In [168]:
print("Missing values:", data.isna().sum().sum())
print("Duplicate rows:", data.duplicated().sum())
print("Duplicate IDs:", data["id"].duplicated().sum())

Missing values: 0
Duplicate rows: 0
Duplicate IDs: 0


In [169]:
print("Age range:", data["age_years"].min(), "to", data["age_years"].max())

print("Age mean:", data["age_years"].mean())

Age range: 29 to 64
Age mean: 52.823634630892165


In [170]:
invalid_bp = data[
    (data["ap_hi"] <= data["ap_lo"])
    | (data["ap_hi"] < 70)
    | (data["ap_hi"] > 250)
    | (data["ap_lo"] < 40)
    | (data["ap_lo"] > 150)
]

print("Suspicious BP records:", len(invalid_bp))

invalid_bp[["id", "age_years", "ap_hi", "ap_lo", "cardio"]]

Suspicious BP records: 5


,id,age_years,ap_hi,ap_lo,cardio
12466,18238,43,100,100,1
22303,32702,56,95,100,1
31366,45975,63,100,100,0
45316,66423,41,90,100,0
51424,75277,57,90,120,1


In [171]:
data = data.drop(index=invalid_bp.index).reset_index(drop=True)

print("Shape after BP cleaning:", data.shape)

Shape after BP cleaning: (68200, 17)


In [172]:
extreme_bmi = data[(data["bmi"] < 10) | (data["bmi"] > 60)]

print("Extreme BMI records:", len(extreme_bmi))

extreme_bmi[["id", "height", "weight", "bmi", "cardio"]].sort_values("bmi")

Extreme BMI records: 65


,id,height,weight,bmi,cardio
32942,48318,178,11.0,3.471784,1
28722,42156,177,22.0,7.022248,0
58639,85931,162,21.0,8.001829,1
26117,38312,157,23.0,9.331007,0
59133,86650,171,29.0,9.917581,1
...,...,...,...,...,...
28398,41661,60,69.0,191.666667,0
23298,34186,81,156.0,237.768633,0
22135,32456,55,81.0,267.768595,1
26679,39156,80,178.0,278.125000,1


In [173]:
invalid_anthropometric = data[
    (data["height"] < 100)
    | (data["height"] > 220)
    | (data["weight"] < 30)
    | (data["weight"] > 250)
]

print("Potentially invalid height/weight records:", len(invalid_anthropometric))

Potentially invalid height/weight records: 33


In [174]:
invalid_anthropometric[
    ["id", "age_years", "height", "weight", "bmi", "cardio"]
].sort_values(["height", "weight"]).head()

,id,age_years,height,weight,bmi,cardio
22135,32456,64,55,81.0,267.768595,1
64934,95141,51,57,61.0,187.750077,1
62465,91523,50,59,57.6,165.469693,0
28398,41661,52,60,69.0,191.666667,0
26891,39462,57,64,61.0,148.925781,0


In [175]:
valid_anthropometric = (
    (data["height"] >= 100)
    & (data["height"] <= 220)
    & (data["weight"] >= 30)
    & (data["weight"] <= 250)
)

print("Rows failing anthropometric checks:", (~valid_anthropometric).sum())

Rows failing anthropometric checks: 33


In [176]:
data = data[valid_anthropometric].reset_index(drop=True)

print("Shape after anthropometric cleaning:", data.shape)

Shape after anthropometric cleaning: (68167, 17)


In [177]:
TARGET = "cardio"

y = data[TARGET].copy()

print(y.value_counts())
print()
print(y.value_counts(normalize=True).mul(100))

cardio
0    34513
1    33654
Name: count, dtype: int64

cardio
0    50.63007
1    49.36993
Name: proportion, dtype: float64


In [ ]:
BASELINE_FEATURES = [
    "age_years",
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
]

In [179]:
ENGINEERED_FEATURES = BASELINE_FEATURES + ["bmi", "bp_category"]

In [180]:
print("Baseline features:")
for feature in BASELINE_FEATURES:
    print(" -", feature)

print("\nEngineered features:")
for feature in ENGINEERED_FEATURES:
    print(" -", feature)

Baseline features:
 - age_years
 - gender
 - height
 - weight
 - ap_hi
 - ap_lo
 - cholesterol
 - gluc
 - smoke
 - alco
 - active

Engineered features:
 - age_years
 - gender
 - height
 - weight
 - ap_hi
 - ap_lo
 - cholesterol
 - gluc
 - smoke
 - alco
 - active
 - bmi
 - bp_category


In [181]:
train_data, test_data = train_test_split(
    data, test_size=0.20, random_state=42, stratify=data[TARGET]
)

In [182]:
print("Train:", train_data.shape)
print("Test :", test_data.shape)

Train: (54533, 17)
Test : (13634, 17)


In [183]:
print("Training target:")
print(train_data[TARGET].value_counts(normalize=True).mul(100))

print("\nTest target:")
print(test_data[TARGET].value_counts(normalize=True).mul(100))

Training target:
cardio
0    50.629894
1    49.370106
Name: proportion, dtype: float64

Test target:
cardio
0    50.630776
1    49.369224
Name: proportion, dtype: float64


In [184]:
X_train_baseline = train_data[BASELINE_FEATURES].copy()
X_test_baseline = test_data[BASELINE_FEATURES].copy()

y_train = train_data[TARGET].copy()
y_test = test_data[TARGET].copy()

In [185]:
X_train_baseline.to_csv(PROCESSED_DIR / "X_train_raw_baseline.csv", index=False)

X_test_baseline.to_csv(PROCESSED_DIR / "X_test_raw_baseline.csv", index=False)

print("Saved raw baseline feature splits.")

Saved raw baseline feature splits.


In [186]:
X_train_engineered = train_data[ENGINEERED_FEATURES].copy()
X_test_engineered = test_data[ENGINEERED_FEATURES].copy()

In [187]:
X_train_engineered.to_csv(PROCESSED_DIR / "X_train_raw_engineered.csv", index=False)

X_test_engineered.to_csv(PROCESSED_DIR / "X_test_raw_engineered.csv", index=False)

print("Saved raw engineered feature splits.")

Saved raw engineered feature splits.


In [188]:
BASELINE_NUMERICAL = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

BASELINE_CATEGORICAL = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

In [ ]:
ENGINEERED_NUMERICAL = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi"]

ENGINEERED_CATEGORICAL = [
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
    "bp_category",
]

In [ ]:
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), BASELINE_NUMERICAL),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            BASELINE_CATEGORICAL,
        ),
    ]
)

In [ ]:
engineered_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ENGINEERED_NUMERICAL),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ENGINEERED_CATEGORICAL,
        ),
    ]
)

In [192]:
X_train_baseline_processed = baseline_preprocessor.fit_transform(X_train_baseline)

X_test_baseline_processed = baseline_preprocessor.transform(X_test_baseline)

In [193]:
X_train_engineered_processed = engineered_preprocessor.fit_transform(X_train_engineered)

X_test_engineered_processed = engineered_preprocessor.transform(X_test_engineered)

In [194]:
print("Baseline train shape:", X_train_baseline_processed.shape)

print("Baseline test shape:", X_test_baseline_processed.shape)

print("Engineered train shape:", X_train_engineered_processed.shape)

print("Engineered test shape:", X_test_engineered_processed.shape)

Baseline train shape: (54533, 19)
Baseline test shape: (13634, 19)
Engineered train shape: (54533, 24)
Engineered test shape: (13634, 24)


In [195]:
baseline_feature_names = baseline_preprocessor.get_feature_names_out()

engineered_feature_names = engineered_preprocessor.get_feature_names_out()

In [196]:
print("Baseline features:")
print(baseline_feature_names)

print("\nEngineered features:")
print(engineered_feature_names)

Baseline features:
['num__age_years' 'num__height' 'num__weight' 'num__ap_hi' 'num__ap_lo'
 'cat__gender_1' 'cat__gender_2' 'cat__cholesterol_1' 'cat__cholesterol_2'
 'cat__cholesterol_3' 'cat__gluc_1' 'cat__gluc_2' 'cat__gluc_3'
 'cat__smoke_0' 'cat__smoke_1' 'cat__alco_0' 'cat__alco_1' 'cat__active_0'
 'cat__active_1']

Engineered features:
['num__age_years' 'num__height' 'num__weight' 'num__ap_hi' 'num__ap_lo'
 'num__bmi' 'cat__gender_1' 'cat__gender_2' 'cat__cholesterol_1'
 'cat__cholesterol_2' 'cat__cholesterol_3' 'cat__gluc_1' 'cat__gluc_2'
 'cat__gluc_3' 'cat__smoke_0' 'cat__smoke_1' 'cat__alco_0' 'cat__alco_1'
 'cat__active_0' 'cat__active_1' 'cat__bp_category_Elevated'
 'cat__bp_category_Hypertension Stage 1'
 'cat__bp_category_Hypertension Stage 2' 'cat__bp_category_Normal']


In [ ]:
X_train_baseline_processed = pd.DataFrame(
    X_train_baseline_processed,
    columns=baseline_feature_names,
    index=X_train_baseline.index,
)

X_test_baseline_processed = pd.DataFrame(
    X_test_baseline_processed,
    columns=baseline_feature_names,
    index=X_test_baseline.index,
)

In [ ]:
X_train_engineered_processed = pd.DataFrame(
    X_train_engineered_processed,
    columns=engineered_feature_names,
    index=X_train_engineered.index,
)

X_test_engineered_processed = pd.DataFrame(
    X_test_engineered_processed,
    columns=engineered_feature_names,
    index=X_test_engineered.index,
)

In [199]:
X_train_baseline_processed.to_csv(PROCESSED_DIR / "X_train_baseline.csv", index=False)

X_test_baseline_processed.to_csv(PROCESSED_DIR / "X_test_baseline.csv", index=False)

X_train_engineered_processed.to_csv(
    PROCESSED_DIR / "X_train_engineered.csv", index=False
)

X_test_engineered_processed.to_csv(PROCESSED_DIR / "X_test_engineered.csv", index=False)

y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)

y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

In [ ]:
joblib.dump(baseline_preprocessor, PROCESSED_DIR / "baseline_preprocessor.joblib")

joblib.dump(engineered_preprocessor, PROCESSED_DIR / "engineered_preprocessor.joblib")

['../data/processed/engineered_preprocessor.joblib']

In [201]:
print("Processed files created:")

for path in sorted(PROCESSED_DIR.iterdir()):
    print(" -", path.name)

Processed files created:
 - X_test_baseline.csv
 - X_test_engineered.csv
 - X_test_raw_baseline.csv
 - X_test_raw_engineered.csv
 - X_train_baseline.csv
 - X_train_engineered.csv
 - X_train_raw_baseline.csv
 - X_train_raw_engineered.csv
 - baseline_preprocessor.joblib
 - engineered_preprocessor.joblib
 - y_test.csv
 - y_train.csv
